In [1]:
# 安装依赖
!pip install transformers datasets accelerate peft bitsandbytes evaluate wandb huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass
import wandb

def get_secret(name: str):
    token = os.environ.get(name)
    if token:
        return token.strip(), "env"
    try:
        from google.colab import userdata
        token = userdata.get(name)
        if token:
            return token.strip(), "colab_secret"
    except Exception as e:
        print(f"读取 Colab Secret '{name}' 失败: {type(e).__name__}: {e}")
    return None, None

wandb_key, wandb_source = get_secret("WANDB_API_KEY")
print("WANDB_API_KEY exists:", bool(wandb_key), f"(source={wandb_source})")
if not wandb_key:
    wandb_key = getpass("请输入 WandB API Key（输入不回显）：").strip()
wandb.login(key=wandb_key)

run = wandb.init(
    project="hf-text-classification",
    name="bert-base-chinese-demo",
    notes="Colab GPU 中文文本分类微调示例",
    config={
        "model_name": "bert-base-chinese",
        "epoch": 3,
        "batch_size": 16,
        "lr": 2e-5,
        "weight_decay": 0.01,
    },
)


读取 Colab Secret 'WANDB_API_KEY' 失败: TimeoutException: Requesting secret WANDB_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.
WANDB_API_KEY exists: False (source=None)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zuiaipixiu (zuiaipixiu-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

def get_hf_token():
    for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        token = os.environ.get(key)
        if token:
            return token.strip(), f"env:{key}"

    try:
        import google.colab  # noqa: F401
        in_colab = True
    except ImportError:
        in_colab = False
    print("Running in Colab:", in_colab)

    if not in_colab:
        return None, None

    from google.colab import userdata
    # 注意：Secret 名必须完全是 HF_TOKEN，且右侧 Notebook access 打开
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        try:
            token = userdata.get(name)
            if token:
                return token.strip(), f"colab_secret:{name}"
            print(f"Secret '{name}' 读到了空值")
        except Exception as e:
            print(f"读取 Secret '{name}' 失败: {type(e).__name__}: {e}")

    print(
        "\n请检查 Colab Secrets：\n"
        "1) 左侧钥匙图标 → Secrets\n"
        "2) Name = HF_TOKEN（完全一致）\n"
        "3) 该行右侧 Notebook access 开关必须打开\n"
        "4) 改完后：运行时 → 重新启动运行时，再跑本 cell\n"
    )
    return None, None

hf_token, source = get_hf_token()
print("HF_TOKEN exists:", bool(hf_token), f"(source={source})")

if not hf_token:
    print("改用手动输入。Token 获取：https://huggingface.co/settings/tokens （勾选 Write）")
    hf_token = getpass("请输入 HF Access Token（输入不回显）：").strip()
    source = "getpass"

hf_token = hf_token.strip().strip('"').strip("'")
if not hf_token.startswith("hf_"):
    raise RuntimeError("Token 格式不对，应以 hf_ 开头。请重新复制。")

login(token=hf_token, add_to_git_credential=False)
info = whoami()
print(f"Hugging Face 登录成功: user={info.get('name')} (source={source})")


Running in Colab: True
读取 Secret 'HF_TOKEN' 失败: TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.
读取 Secret 'HUGGING_FACE_HUB_TOKEN' 失败: TimeoutException: Requesting secret HUGGING_FACE_HUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

请检查 Colab Secrets：
1) 左侧钥匙图标 → Secrets
2) Name = HF_TOKEN（完全一致）
3) 该行右侧 Notebook access 开关必须打开
4) 改完后：运行时 → 重新启动运行时，再跑本 cell

HF_TOKEN exists: False (source=None)
改用手动输入。Token 获取：https://huggingface.co/settings/tokens （勾选 Write）
Hugging Face 登录成功: user=zuiaipixiu (source=getpass)


In [4]:
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import torch

dataset = load_dataset("lansinuote/ChnSentiCorp")

model_name = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def load_bert_for_classification(model_name: str, num_labels: int = 2):
    """兼容 LayerNorm gamma/beta 与 weight/bias 命名差异。"""
    config = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
    model = AutoModelForSequenceClassification.from_config(config)

    try:
        from safetensors.torch import load_file
        from huggingface_hub import hf_hub_download
        weight_path = hf_hub_download(repo_id=model_name, filename="model.safetensors")
        state_dict = load_file(weight_path)
    except Exception:
        from huggingface_hub import hf_hub_download
        weight_path = hf_hub_download(repo_id=model_name, filename="pytorch_model.bin")
        state_dict = torch.load(weight_path, map_location="cpu", weights_only=True)

    remapped = {}
    for key, value in state_dict.items():
        new_key = key.replace("LayerNorm.gamma", "LayerNorm.weight").replace(
            "LayerNorm.beta", "LayerNorm.bias"
        )
        if new_key.startswith("classifier."):
            continue
        remapped[new_key] = value

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    real_missing = [k for k in missing if not k.startswith("classifier.")]
    if real_missing:
        raise RuntimeError("BERT 主干未完整加载: " + ", ".join(real_missing[:8]))
    ln = model.bert.embeddings.LayerNorm
    print("LayerNorm check: weight_mean=", float(ln.weight.mean()), "bias_mean=", float(ln.bias.mean()))
    print("BERT 主干权重加载成功（classifier 为新建头，属正常）")
    return model


model = load_bert_for_classification(model_name, num_labels=2)

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA GPU，请在 Colab 中选择：运行时 -> 更改运行时类型 -> GPU")

device = torch.device("cuda")
model.to(device)
print("Using GPU:", torch.cuda.get_device_name(0))

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_fn, batched=True)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)


dataset_infos.json:   0%|          | 0.00/960 [00:00<?, ?B/s]

data/train-00000-of-00001-02f200ca5f2a78(…): reconstructing file:   0%|          |  0.00B / 2.16MB            

data/train-00000-of-00001-02f200ca5f2a78(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-405befbaa(…): reconstructing file:   0%|          |  0.00B /  276kB            

data/validation-00000-of-00001-405befbaa(…): downloading bytes:           |  0.00B            

...

/tmp/ipykernel_955/1616422739.py:47: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("LayerNorm check: weight_mean=", float(ln.weight.mean()), "bias_mean=", float(ln.bias.mean()))


LayerNorm check: weight_mean= 0.886962890625 bias_mean= -0.0143721429631114
BERT 主干权重加载成功（classifier 为新建头，属正常）
Using GPU: Tesla T4


Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [5]:
training_args = TrainingArguments(
    output_dir="./bert-senti-demo",  # 本地临时保存权重
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,        # 每10步记录日志到wandb
    eval_strategy="epoch",  # 每个epoch结束验证
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,        # 训练完成自动上传到HF Hub
    hub_model_id="zuiaipixiu/bert-senti-colab-demo",  # 替换为你的HF仓库名
    report_to="wandb",       # 核心：对接wandb记录实验
    fp16=True,  # GPU半精度加速（Colab T4支持）
)

In [6]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# 开始训练，loss/acc自动实时同步到WandB网页看板
trainer.train()

# 在测试集评估
test_result = trainer.evaluate(tokenized_dataset["test"])
print("测试集指标：", test_result)

# 将测试结果也记录到wandb
wandb.log(test_result)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.286455,0.215398,0.925000
2,0.207835,0.224298,0.935833
3,0.074318,0.265787,0.942500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.074318,0.200801,3,0.930000


测试集指标： {'eval_loss': 0.20080111920833588, 'eval_accuracy': 0.93}


In [7]:
wandb.finish()
print("WandB实验记录已保存，可打开链接查看完整曲线、超参、指标")

eval/accuracy,▁▅█▃
eval/loss,▃▄█▁
eval/runtime,▁▆█▄
eval/samples_per_second,█▃▁▅
eval/steps_per_second,█▃▁▅
eval_accuracy,▁
eval_loss,▁
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇██
train/global_step,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▂▃▄▂▃▄▃▅▂▃▃▂▁▃▁▁▁▁▁▄▄▁▃█▁▃▁▃▃▂▂▁▁▁▂▁▅▁▁▄
+2,...


WandB实验记录已保存，可打开链接查看完整曲线、超参、指标
